# Exp52, Hierarchical Group Signal Recovery

Final experiment for the competition. This builds on the strongest local recipe, Exp43, instead of repeating the failed CatBoost direction from Exp51.

**Idea:** keep the exact-value, digit, frequency, and neighbourhood features, then add leakage-safe target rates for repeated row/group signatures. Every target-derived feature is calculated inside each training fold and mapped onto validation/test, so the validation score stays honest.

The goal is simple: recover synthetic-generator structure that ordinary trees may only see indirectly.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from scipy.stats import rankdata

SEED = 52
N_FOLDS = 5
DATA_DIR = Path("data")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
RESULTS_DIR = Path("results")
SUB_DIR = Path("submissions")
RESULTS_DIR.mkdir(exist_ok=True)
SUB_DIR.mkdir(exist_ok=True)

print("Exp52 loaded")


In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
y = train["Will_Buy_EV"].map({"Yes": 1, "No": 0}).astype(np.int8).to_numpy()

TARGET = "Will_Buy_EV"
ID = "id"
NUMERIC = [
    "Age", "Annual_Income_USD", "Daily_Commute_km",
    "Number_of_Cars_Owned", "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work", "Environmental_Concern_Level",
]
CATEGORICAL = [
    "Gender", "City_Type", "Current_Car_Type",
    "Home_Charging_Possible", "Subsidy_Available", "Range_Anxiety_Level",
]
BASE = NUMERIC + CATEGORICAL

print("Train:", train.shape, "Test:", test.shape)
print("Positive rate:", round(y.mean(), 6))


## Feature construction

The base features deliberately stay close to Exp43. The new part is the hierarchical group layer added later.

In [ ]:
def add_static_features(df):
    x = df[BASE].copy()

    for c in CATEGORICAL:
        x[c] = x[c].fillna("__NA__").astype(str)

    eps = 1e-3
    x["income_per_age"] = x["Annual_Income_USD"] / (x["Age"] + eps)
    x["income_per_commute"] = x["Annual_Income_USD"] / (x["Daily_Commute_km"] + eps)
    x["commute_per_age"] = x["Daily_Commute_km"] / (x["Age"] + eps)
    x["income_per_car"] = x["Annual_Income_USD"] / (x["Number_of_Cars_Owned"] + 1.0)
    x["income_per_home_station"] = x["Annual_Income_USD"] / (x["Charging_Stations_Near_Home"] + 1.0)
    x["income_per_work_station"] = x["Annual_Income_USD"] / (x["Charging_Stations_Near_Work"] + 1.0)
    x["total_stations"] = x["Charging_Stations_Near_Home"] + x["Charging_Stations_Near_Work"]
    x["station_gap"] = x["Charging_Stations_Near_Home"] - x["Charging_Stations_Near_Work"]
    x["income_x_concern"] = x["Annual_Income_USD"] * x["Environmental_Concern_Level"]
    x["commute_x_concern"] = x["Daily_Commute_km"] * x["Environmental_Concern_Level"]
    x["income_x_subsidy"] = x["Annual_Income_USD"] * (x["Subsidy_Available"] == "Yes").astype(np.int8)
    x["concern_x_subsidy"] = x["Environmental_Concern_Level"] * (x["Subsidy_Available"] == "Yes").astype(np.int8)

    pairs = {
        "City_Car": ["City_Type", "Current_Car_Type"],
        "City_Subsidy": ["City_Type", "Subsidy_Available"],
        "City_HomeCharge": ["City_Type", "Home_Charging_Possible"],
        "Car_Subsidy": ["Current_Car_Type", "Subsidy_Available"],
        "Car_HomeCharge": ["Current_Car_Type", "Home_Charging_Possible"],
        "Anxiety_Subsidy": ["Range_Anxiety_Level", "Subsidy_Available"],
        "Anxiety_HomeCharge": ["Range_Anxiety_Level", "Home_Charging_Possible"],
    }
    for name, cols in pairs.items():
        x[name] = x[cols].astype(str).agg("__".join, axis=1)

    # Exact numeric decomposition, matching the strongest part of the earlier recipe.
    digit_cols = [c for c in NUMERIC if c != "Environmental_Concern_Level"]
    for c in digit_cols:
        vals = pd.to_numeric(x[c], errors="coerce").fillna(-999).astype(np.int64)
        for p in (1, 10, 100, 1000):
            x[f"{c}_digit_{p}"] = ((vals // p) % 10).astype(np.int16)

    # Small deterministic bins for local tree splits.
    x["Income_Bin_5k"] = (x["Annual_Income_USD"] // 5000).astype(np.int32)
    x["Income_Bin_10k"] = (x["Annual_Income_USD"] // 10000).astype(np.int32)
    x["Commute_Bin_2"] = (x["Daily_Commute_km"] // 2).astype(np.int32)
    x["Commute_Bin_5"] = (x["Daily_Commute_km"] // 5).astype(np.int32)
    x["Age_Bin_2"] = (x["Age"] // 2).astype(np.int32)
    x["Age_Bin_5"] = (x["Age"] // 5).astype(np.int32)

    return x

X_all = add_static_features(train)
T_all = add_static_features(test)

# Encode categorical/static string columns consistently for XGBoost.
ALL = pd.concat([X_all, T_all], axis=0, ignore_index=True)
for c in ALL.columns:
    if ALL[c].dtype == "object":
        ALL[c] = pd.factorize(ALL[c], sort=True)[0].astype(np.int32)
X_all = ALL.iloc[:len(train)].reset_index(drop=True)
T_all = ALL.iloc[len(train):].reset_index(drop=True)

print("Static feature count:", X_all.shape[1])


## Leakage-safe frequency + hierarchical target encoding

This is the new Exp52 component. The same raw row can appear many times in synthetic data, and several partial signatures can repeat even when the full row does not. For each fold, the validation row only sees statistics computed from the other four training folds.

In [ ]:
RAW_TRAIN = train[BASE].copy()
RAW_TEST = test[BASE].copy()
for c in CATEGORICAL:
    RAW_TRAIN[c] = RAW_TRAIN[c].fillna("__NA__").astype(str)
    RAW_TEST[c] = RAW_TEST[c].fillna("__NA__").astype(str)

GROUPS = {
    "full_row": BASE,
    "core_numeric": ["Annual_Income_USD", "Daily_Commute_km", "Environmental_Concern_Level", "Subsidy_Available"],
    "env_policy": ["Environmental_Concern_Level", "Subsidy_Available", "Range_Anxiety_Level", "Home_Charging_Possible"],
    "income_context": ["Annual_Income_USD", "Daily_Commute_km", "Charging_Stations_Near_Home", "Charging_Stations_Near_Work"],
    "city_policy": ["City_Type", "Subsidy_Available", "Home_Charging_Possible", "Current_Car_Type"],
    "mobility": ["Current_Car_Type", "Range_Anxiety_Level", "Daily_Commute_km", "Number_of_Cars_Owned"],
}

def make_key(df, cols):
    return df[cols].astype(str).agg("|".join, axis=1)

def add_fold_group_features(Xtr, Xva, Tst, raw_tr, raw_va, raw_te, ytr):
    prior = float(ytr.mean())
    out_tr = Xtr.copy()
    out_va = Xva.copy()
    out_te = Tst.copy()

    # Label-free frequencies. These use the available feature values only.
    for c in BASE:
        vc = pd.concat(
            [raw_tr[c], raw_va[c], raw_te[c]],
            ignore_index=True
        ).value_counts(dropna=False)

        denom = len(raw_tr) + len(raw_va) + len(raw_te)

        out_tr[f"freq_{c}"] = (
            raw_tr[c].map(vc).fillna(0).to_numpy(dtype=np.float32) / denom
        )
        out_va[f"freq_{c}"] = (
            raw_va[c].map(vc).fillna(0).to_numpy(dtype=np.float32) / denom
        )
        out_te[f"freq_{c}"] = (
            raw_te[c].map(vc).fillna(0).to_numpy(dtype=np.float32) / denom
        )

    for name, cols in GROUPS.items():
        ktr = make_key(raw_tr, cols)
        kva = make_key(raw_va, cols)
        kte = make_key(raw_te, cols)

        stats = (
            pd.DataFrame({
                "k": ktr.to_numpy(),
                "y": ytr
            })
            .groupby("k")["y"]
            .agg(["sum", "count"])
        )

        smooth = 12.0 if name in ("full_row", "core_numeric") else 25.0

        # ---------------------------------------------------------
        # IMPORTANT:
        # Training rows use leave-one-out statistics.
        # This prevents a row from seeing its own target.
        # ---------------------------------------------------------
        train_sum_loo = (
            ktr.map(stats["sum"]).to_numpy(dtype=np.float64) - ytr
        )
        train_count_loo = (
            ktr.map(stats["count"]).to_numpy(dtype=np.float64) - 1.0
        )

        train_rate = (
            (train_sum_loo + prior * smooth)
            / (train_count_loo + smooth)
        ).astype(np.float32)

        # Validation and test only use the training-fold statistics.
        val_sum = kva.map(stats["sum"])
        val_count = kva.map(stats["count"])

        test_sum = kte.map(stats["sum"])
        test_count = kte.map(stats["count"])

        val_rate = (
            (val_sum.fillna(0).to_numpy(dtype=np.float64)
             + prior * smooth)
            /
            (val_count.fillna(0).to_numpy(dtype=np.float64)
             + smooth)
        ).astype(np.float32)

        test_rate = (
            (test_sum.fillna(0).to_numpy(dtype=np.float64)
             + prior * smooth)
            /
            (test_count.fillna(0).to_numpy(dtype=np.float64)
             + smooth)
        ).astype(np.float32)

        out_tr[f"te_{name}"] = train_rate
        out_va[f"te_{name}"] = val_rate
        out_te[f"te_{name}"] = test_rate

        # Frequency counts are also leave-one-out for training.
        out_tr[f"freq_group_{name}"] = np.maximum(
            train_count_loo, 0
        ).astype(np.float32)

        out_va[f"freq_group_{name}"] = (
            val_count.fillna(0).to_numpy(dtype=np.float32)
        )

        out_te[f"freq_group_{name}"] = (
            test_count.fillna(0).to_numpy(dtype=np.float32)
        )

    return out_tr, out_va, out_te


In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_a = np.zeros(len(train), dtype=np.float64)
oof_b = np.zeros(len(train), dtype=np.float64)
test_a = np.zeros(len(test), dtype=np.float64)
test_b = np.zeros(len(test), dtype=np.float64)
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_all, y), 1):
    print(f"\n========== FOLD {fold}/{N_FOLDS} ==========")
    Xtr0, Xva0 = X_all.iloc[tr_idx].copy(), X_all.iloc[va_idx].copy()
    ytr, yva = y[tr_idx], y[va_idx]
    raw_tr, raw_va = RAW_TRAIN.iloc[tr_idx], RAW_TRAIN.iloc[va_idx]

    Xtr, Xva, Tst = add_fold_group_features(Xtr0, Xva0, T_all.copy(), raw_tr, raw_va, RAW_TEST, ytr)

    common = dict(
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        max_bin=256,
        subsample=0.88,
        colsample_bytree=0.88,
        min_child_weight=3,
        reg_alpha=0.05,
        reg_lambda=2.0,
        n_jobs=-1,
    )

    model_a = XGBClassifier(max_depth=6, learning_rate=0.028, n_estimators=2600, random_state=SEED + fold, **common)
    model_b = XGBClassifier(max_depth=7, learning_rate=0.022, n_estimators=3000, random_state=SEED + fold + 100, **common)

    model_a.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    model_b.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)

    pa = model_a.predict_proba(Xva)[:, 1]
    pb = model_b.predict_proba(Xva)[:, 1]
    ta = model_a.predict_proba(Tst)[:, 1]
    tb = model_b.predict_proba(Tst)[:, 1]

    oof_a[va_idx] = pa
    oof_b[va_idx] = pb
    test_a += ta / N_FOLDS
    test_b += tb / N_FOLDS

    sa = roc_auc_score(yva, pa)
    sb = roc_auc_score(yva, pb)
    blend = 0.65 * rankdata(pa) + 0.35 * rankdata(pb)
    blend_score = roc_auc_score(yva, blend)
    fold_scores.append((fold, sa, sb, blend_score))
    print(f"A={sa:.6f}  B={sb:.6f}  rank blend={blend_score:.6f}")

print("\nFold summary:")
print(pd.DataFrame(fold_scores, columns=["fold","model_a","model_b","rank_blend"]))
print("OOF A:", roc_auc_score(y, oof_a))
print("OOF B:", roc_auc_score(y, oof_b))


## In-memory blend search

Only the best final recipe is written out. No giant experiment CSVs are generated.

In [ ]:
recipes = []
rank_a = rankdata(oof_a) / len(oof_a)
rank_b = rankdata(oof_b) / len(oof_b)
rank_ta = rankdata(test_a) / len(test_a)
rank_tb = rankdata(test_b) / len(test_b)

for w in np.arange(0.0, 1.01, 0.05):
    p = w * oof_a + (1 - w) * oof_b
    pt = w * test_a + (1 - w) * test_b
    recipes.append((roc_auc_score(y, p), f"raw_{w:.2f}", w))
    rp = w * rank_a + (1 - w) * rank_b
    rt = w * rank_ta + (1 - w) * rank_tb
    recipes.append((roc_auc_score(y, rp), f"rank_{w:.2f}", w))

recipes = sorted(recipes, reverse=True)
best_auc, best_name, best_w = recipes[0]
print("Best OOF recipe:", best_name, "AUC:", round(best_auc, 7))

# Save only compact recipe metadata.
pd.DataFrame(recipes[:20], columns=["oof_auc","recipe","weight_a"]).to_csv(RESULTS_DIR / "exp52_top_recipes.csv", index=False)

if best_name.startswith("rank"):
    final_test = best_w * rank_ta + (1 - best_w) * rank_tb
else:
    final_test = best_w * test_a + (1 - best_w) * test_b

submission = pd.DataFrame({ID: test[ID], TARGET: final_test})
submission_path = SUB_DIR / "submission_16_exp52.csv"
submission.to_csv(submission_path, index=False)

print("\nFINAL")
print("OOF:", round(best_auc, 7))
print("Recipe:", best_name)
print("Submission:", submission_path)
print("Rows:", len(submission))


## What to submit

If the OOF is clearly above the Exp43 baseline of **0.945761**, this is the final model to test. If it is below that line, do not pretend the new feature layer helped. The notebook still preserves the result for comparison.